In [1]:
import sys
import numpy as np
import pybullet as p
import time
import logging
from typing import List, Tuple, Optional, Sequence, Collection, Dict, Any, cast, Set
import random
import json

from predicators.structs import Action, Array, GroundAtom, Object, State, Type, ParameterizedOption, EnvironmentTask,\
    Image, Predicate, State, Type, Video
from predicators import utils
from predicators.settings import CFG
from gymnasium.spaces import Box

#Import core environment methods, robot function etc.

from predicators.envs.pybullet_blocks import PyBulletBlocksEnv
from predicators.envs.pybullet_env import PyBulletEnv, create_pybullet_block
from predicators.pybullet_helpers.robots import SingleArmPyBulletRobot
from predicators.pybullet_helpers.geometry import Pose
from predicators.pybullet_helpers.joint import JointPositions, get_joint_infos, get_joint_positions
from predicators.pybullet_helpers.link import get_link_state

#Import the functions that are to be tested:

from predicators.pybullet_helpers.motion_planning import run_motion_planning
#The pick/place options to be tested are accessed via the env instance
from predicators.pybullet_helpers.controllers import create_move_end_effector_to_pose_option,\
                                                    create_change_fingers_option

import copy

import matplotlib
import PIL
from PIL import ImageDraw

try:
    import gymnasium as mujoco_kitchen_gym
    from gymnasium_robotics.utils.mujoco_utils import get_joint_qpos, \
        get_site_xmat, get_site_xpos
    from gymnasium_robotics.utils.rotations import mat2quat
    _MJKITCHEN_IMPORTED = True
except (ImportError, RuntimeError):
    _MJKITCHEN_IMPORTED = False
from predicators.envs import BaseEnv
from predicators.envs.kitchen import KitchenEnv

#Configure logging for better debugging outputs:
#logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.basicConfig(
    level=logging.WARNING,                    
    format="%(asctime)s %(name)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

#Defining test configuration, and overriding some default ones:
CFG.use_gui = True

CFG.seed = random.randint(0,10000)

CFG.kitchen_use_perfect_samplers = True
CFG.kitchen_goals = "knob_only"
#Num of PyBullet physics steps per high-level Action in visualize_action_sequence
CFG.pybullet_sim_steps_per_action = 20

# 提高分辨率
CFG.pybullet_camera_width = 1674  # 从335提高到1674
CFG.pybullet_camera_height = 900  # 从180提高到900

# 提高DPI
CFG.render_state_dpi = 300  # 从150提高到300

# 提高帧率
CFG.video_fps = 10  # 从2提高到10

CFG.make_test_videos = False
CFG.make_failure_videos = False

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\12058\.conda\envs\predicators\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\12058\.conda\envs\predicators\lib\site-packages\scipy\__init__.py:169: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
# 使用最简单的方法：直接调用reset然后只渲染
print("正在创建Mujoco Kitchen环境...")

# 导入gymnasium
import gymnasium as mujoco_kitchen_gym

# 直接创建gym环境
gym_env = mujoco_kitchen_gym.make("FrankaKitchen-v1", 
                                  render_mode="human",
                                  ik_controller=True)

print("Mujoco Kitchen环境创建成功！")

# 直接调用reset来满足gymnasium的要求
print("正在调用reset以满足渲染要求...")
obs, info = gym_env.reset()
print("Reset完成，现在可以渲染了")
print("注意：环境已重置但不会执行任务，只进行渲染")

print("Mujoco环境已打开。按Ctrl+C退出。")
print("注意：环境已准备就绪，只进行渲染。")

try:
    step_count = 0
    print("开始渲染循环...")
    while True:
        # 直接渲染gym环境，不执行任何动作
        gym_env.render()
        
        # 每100步打印一次状态
        if step_count % 100 == 0:
            print(f"环境渲染中... 步骤: {step_count}")
        
        step_count += 1
        time.sleep(0.1)  # 控制渲染频率
except KeyboardInterrupt:
    print("\n正在关闭环境...")
    print("感谢使用Mujoco Kitchen环境查看器！")
except Exception as e:
    print(f"运行时出错: {e}")
    print(f"错误类型: {type(e)}")
    print(f"环境类型: {type(gym_env)}")

正在创建Mujoco Kitchen环境...


Mujoco Kitchen环境创建成功！
正在调用reset以满足渲染要求...
Reset完成，现在可以渲染了
注意：环境已重置但不会执行任务，只进行渲染
Mujoco环境已打开。按Ctrl+C退出。
注意：环境已准备就绪，只进行渲染。
开始渲染循环...
环境渲染中... 步骤: 0
Pressed ESC
Quitting.


Exception ignored on calling ctypes callback function: <function _handle_glfw_errors at 0x000001C2FDA7BE20>
Traceback (most recent call last):
  File "c:\Users\12058\.conda\envs\predicators\lib\site-packages\glfw\__init__.py", line 662, in callback_wrapper
    @functools.wraps(func)
  File "_pydevd_bundle\\pydevd_cython.pyx", line 1697, in _pydevd_bundle.pydevd_cython.SafeCallWrapper.__call__
  File "_pydevd_bundle\\pydevd_cython.pyx", line 2017, in _pydevd_bundle.pydevd_cython.ThreadTracer.__call__
  File "c:\Users\12058\.conda\envs\predicators\lib\site-packages\debugpy\_vendored\pydevd\_pydev_bundle\pydev_is_thread_alive.py", line 16, in is_thread_alive
    def is_thread_alive(t):
KeyboardInterrupt: 


In [ ]:
# GUI环境验证和交互处理
if 'env' in locals() and 'initial_state' in locals():
    print("=== GUI环境验证 ===")
    
    try:
        # 测试基本状态获取
        current_state = env.state_info_to_state(initial_state["state_info"])
        print(f"✓ 状态获取成功，包含 {len(current_state)} 个物件")
        
        # 测试物件访问
        gripper = env.object_name_to_object("gripper")
        kettle = env.object_name_to_object("kettle")
        print(f"✓ 物件访问成功: {gripper.name}, {kettle.name}")
        
        # 测试动作空间
        action_space = env.action_space
        print(f"✓ 动作空间: {action_space.shape}")
        
        # 测试任务
        task_desc = env._current_task.goal_description
        print(f"✓ 任务描述: {task_desc}")
        
        print("\n🎉 GUI环境验证完成！")
        
        # 显示一些重要物件的位置
        print(f"\n=== 重要物件位置 ===")
        important_objects = ["gripper", "kettle", "knob1", "knob2", "knob3", "knob4"]
        for obj_name in important_objects:
            if obj_name in [obj.name for obj in current_state.keys()]:
                obj = next(obj for obj in current_state.keys() if obj.name == obj_name)
                x = current_state[obj].get("x", "N/A")
                y = current_state[obj].get("y", "N/A") 
                z = current_state[obj].get("z", "N/A")
                print(f"  {obj_name}: ({x:.3f}, {y:.3f}, {z:.3f})")
        
        print(f"\n=== GUI窗口状态 ===")
        print("🎮 GUI窗口应该已经打开")
        print("💡 如果窗口无响应，请尝试：")
        print("   1. 点击GUI窗口使其获得焦点")
        print("   2. 等待几秒钟让窗口完全加载")
        print("   3. 如果仍然无响应，重新运行环境创建cell")
        
    except Exception as e:
        print(f"❌ 环境验证失败: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("❌ 环境未正确初始化")


=== 快速环境验证 ===
❌ 环境验证失败: object of type 'State' has no len()


Traceback (most recent call last):
  File "C:\Users\12058\AppData\Local\Temp\ipykernel_354204\1982866544.py", line 8, in <module>
    print(f"✓ 状态获取成功，包含 {len(current_state)} 个物件")
TypeError: object of type 'State' has no len()


In [ ]:
# GUI窗口交互和事件处理
if 'env' in locals():
    print("=== GUI窗口交互处理 ===")
    print("🎮 GUI窗口已打开！现在你可以：")
    print("   - 拖动鼠标来旋转视角")
    print("   - 使用鼠标滚轮来缩放")
    print("   - 右键拖动来平移视角")
    print("   - 查看厨房环境中的各种物件")
    
    # 尝试处理GUI事件
    try:
        print("\n🔄 正在处理GUI事件...")
        # 这里可以添加一些GUI事件处理代码
        # 但通常Mujoco的GUI会自动处理事件
        
        # 显示环境中的物件列表
        if 'initial_state' in locals():
            current_state = env.state_info_to_state(initial_state["state_info"])
            print(f"\n📋 环境中的物件列表:")
            for i, (obj, properties) in enumerate(current_state.items(), 1):
                print(f"   {i:2d}. {obj.name} ({obj.type.name})")
        
        print(f"\n🔧 环境信息:")
        print(f"   - 环境类型: {env.get_name()}")
        print(f"   - 动作空间维度: {env.action_space.shape}")
        print(f"   - 当前任务: {env._current_task.goal_description}")
        
        print(f"\n💡 GUI交互提示：")
        print(f"   - 如果GUI窗口无响应，请点击窗口使其获得焦点")
        print(f"   - 如果窗口卡住，可以重新运行环境创建cell")
        print(f"   - GUI窗口通常会在notebook kernel重启时自动关闭")
        
    except Exception as e:
        print(f"⚠️ GUI事件处理出现问题: {e}")
        print("但环境功能应该正常")
    
else:
    print("❌ 环境未初始化，请先运行环境创建cell")


In [ ]:
# 测试GUI窗口响应性
if 'env' in locals():
    print("=== GUI窗口响应性测试 ===")
    
    try:
        # 测试环境基本功能
        print("1. 测试环境基本功能...")
        current_state = env.state_info_to_state(initial_state["state_info"])
        print(f"   ✓ 状态获取成功，包含 {len(current_state)} 个物件")
        
        # 测试动作空间
        print("2. 测试动作空间...")
        action_space = env.action_space
        print(f"   ✓ 动作空间: {action_space.shape}")
        
        # 测试任务
        print("3. 测试任务...")
        task_desc = env._current_task.goal_description
        print(f"   ✓ 任务描述: {task_desc}")
        
        print("\n🎉 环境功能测试完成！")
        
        # GUI窗口状态检查
        print(f"\n=== GUI窗口状态检查 ===")
        print("🎮 GUI窗口应该已经打开")
        print("💡 请检查GUI窗口是否：")
        print("   - 窗口是否可见？")
        print("   - 窗口是否响应鼠标操作？")
        print("   - 是否可以拖动视角？")
        
        print(f"\n⚠️  如果GUI窗口无响应：")
        print("   1. 点击GUI窗口使其获得焦点")
        print("   2. 等待几秒钟让窗口完全加载")
        print("   3. 尝试拖动鼠标查看环境")
        print("   4. 如果仍然无响应，重新运行环境创建cell")
        
        # 显示环境状态摘要
        print(f"\n📊 环境状态摘要:")
        print(f"   - 物件总数: {len(current_state)}")
        print(f"   - 任务描述: {env._current_task.goal_description}")
        print(f"   - 目标是否达成: {env.goal_reached()}")
        
    except Exception as e:
        print(f"❌ GUI窗口测试失败: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("❌ 环境未初始化")


In [ ]:
# 查看环境中的物件信息
if 'env' in locals():
    print("=== Kitchen环境物件信息 ===")
    
    # 获取当前状态
    current_state = env.state_info_to_state(initial_state["state_info"])
    print(f"状态中的物件数量: {len(current_state)}")
    
    # 列出所有物件及其属性
    print("\n=== 物件详细信息 ===")
    for obj, properties in current_state.items():
        print(f"\n物件: {obj.name} (类型: {obj.type})")
        for prop_name, prop_value in properties.items():
            if isinstance(prop_value, (int, float)):
                print(f"  {prop_name}: {prop_value:.4f}")
            else:
                print(f"  {prop_name}: {prop_value}")
    
    # 显示环境类型信息
    print(f"\n=== 环境类型信息 ===")
    print(f"支持的类型: {[t.name for t in env.types]}")
    
    # 显示谓词信息
    print(f"\n=== 谓词信息 ===")
    print(f"可用谓词: {[p.name for p in env.predicates]}")
    
    # 显示目标谓词
    print(f"\n=== 目标谓词 ===")
    print(f"目标谓词: {[p.name for p in env.goal_predicates]}")
    
    # 显示当前任务
    print(f"\n=== 当前任务 ===")
    print(f"任务描述: {env._current_task.goal_description}")
    
else:
    print("环境未初始化，请先运行上一个cell")


In [ ]:
# 可视化环境 - 安全版本
if 'env' in locals():
    print("=== 环境可视化 ===")
    
    # 检查是否是无GUI模式
    if not CFG.use_gui:
        print("⚠ 当前是无GUI模式，无法渲染图像")
        print("但环境功能完全正常，可以查看物件信息")
        
        # 显示环境状态信息
        if 'initial_state' in locals():
            current_state = env.state_info_to_state(initial_state["state_info"])
            print(f"\n环境状态摘要:")
            print(f"- 物件数量: {len(current_state)}")
            print(f"- 任务描述: {env._current_task.goal_description}")
            print(f"- 动作空间: {env.action_space.shape}")
            
            # 显示重要物件位置
            important_objects = ["gripper", "kettle", "knob1", "knob2", "knob3", "knob4"]
            print(f"\n重要物件位置:")
            for obj_name in important_objects:
                if obj_name in [obj.name for obj in current_state.keys()]:
                    obj = next(obj for obj in current_state.keys() if obj.name == obj_name)
                    x = current_state[obj].get("x", "N/A")
                    y = current_state[obj].get("y", "N/A") 
                    z = current_state[obj].get("z", "N/A")
                    print(f"  {obj_name}: ({x:.3f}, {y:.3f}, {z:.3f})")
        
    else:
        # GUI模式下的渲染
        try:
            video_frames = env.render()
            print(f"渲染成功，获得 {len(video_frames) if video_frames else 0} 帧图像")
            
            if video_frames and len(video_frames) > 0 and video_frames[0] is not None:
                import matplotlib.pyplot as plt
                
                # 显示第一帧图像
                plt.figure(figsize=(12, 8))
                plt.imshow(video_frames[0])
                plt.title("Kitchen Environment - 当前状态")
                plt.axis('off')
                plt.show()
                
                print(f"图像尺寸: {video_frames[0].shape}")
            else:
                print("无法获取有效的图像数据")
                
        except Exception as e:
            print(f"渲染失败: {e}")
            print("这可能是GUI渲染问题，但环境功能正常")
        
else:
    print("环境未初始化，请先运行前面的cell")


In [ ]:
# 测试环境基本功能
if 'env' in locals():
    print("=== 环境功能测试 ===")
    
    # 测试动作空间
    print(f"动作空间: {env.action_space}")
    print(f"动作空间形状: {env.action_space.shape}")
    print(f"动作空间范围: [{env.action_space.low[0]:.3f}, {env.action_space.high[0]:.3f}]")
    
    # 测试随机动作
    print("\n=== 测试随机动作 ===")
    try:
        # 生成随机动作
        random_action = env.action_space.sample()
        print(f"随机动作: {random_action[:5]}... (显示前5个维度)")
        
        # 执行动作
        print("执行随机动作...")
        new_obs = env.step(Action(random_action))
        print("动作执行成功！")
        
        # 检查目标是否达成
        goal_reached = env.goal_reached()
        print(f"目标是否达成: {goal_reached}")
        
    except Exception as e:
        print(f"动作执行失败: {e}")
        import traceback
        traceback.print_exc()
    
    # 显示一些重要的物件位置
    print("\n=== 重要物件位置 ===")
    current_state = env.state_info_to_state(initial_state["state_info"])
    
    important_objects = ["gripper", "kettle", "knob1", "knob2", "knob3", "knob4", "light"]
    for obj_name in important_objects:
        if obj_name in [obj.name for obj in current_state.keys()]:
            obj = next(obj for obj in current_state.keys() if obj.name == obj_name)
            x = current_state[obj].get("x", "N/A")
            y = current_state[obj].get("y", "N/A") 
            z = current_state[obj].get("z", "N/A")
            print(f"{obj_name}: 位置({x:.3f}, {y:.3f}, {z:.3f})")
        else:
            print(f"{obj_name}: 未找到")
            
else:
    print("环境未初始化，请先运行第一个cell")


In [ ]:
# 简化的环境测试
if 'env' in locals():
    print("=== 环境功能验证 ===")
    
    try:
        # 测试基本功能
        print("1. 测试状态获取...")
        current_state = env.state_info_to_state(initial_state["state_info"])
        print(f"   ✓ 状态包含 {len(current_state)} 个物件")
        
        print("2. 测试物件访问...")
        gripper = env.object_name_to_object("gripper")
        kettle = env.object_name_to_object("kettle")
        print(f"   ✓ 成功获取物件: {gripper.name}, {kettle.name}")
        
        print("3. 测试谓词检查...")
        kettle_on_burner1 = env._OnTop_holds(current_state, [kettle, env.object_name_to_object("burner1")])
        print(f"   ✓ 壶在炉子1上: {kettle_on_burner1}")
        
        print("4. 测试动作空间...")
        action_space = env.action_space
        print(f"   ✓ 动作空间形状: {action_space.shape}")
        
        print("5. 测试任务信息...")
        task_desc = env._current_task.goal_description
        print(f"   ✓ 当前任务: {task_desc}")
        
        print("\n🎉 环境功能验证完成！所有基本功能正常")
        
    except Exception as e:
        print(f"❌ 环境功能验证失败: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("❌ 环境未初始化，请先运行前面的cell")


In [ ]:
# 物件交互示例
if 'env' in locals():
    print("=== 物件交互示例 ===")
    
    # 获取当前状态
    current_state = env.state_info_to_state(initial_state["state_info"])
    
    # 检查一些重要的谓词状态
    print("\n=== 谓词状态检查 ===")
    
    # 获取重要物件
    gripper = env.object_name_to_object("gripper")
    kettle = env.object_name_to_object("kettle")
    knob1 = env.object_name_to_object("knob1")
    knob2 = env.object_name_to_object("knob2")
    knob3 = env.object_name_to_object("knob3")
    knob4 = env.object_name_to_object("knob4")
    light = env.object_name_to_object("light")
    burner1 = env.object_name_to_object("burner1")
    burner2 = env.object_name_to_object("burner2")
    burner3 = env.object_name_to_object("burner3")
    burner4 = env.object_name_to_object("burner4")
    
    # 检查壶是否在炉子上
    kettle_on_burner1 = env._OnTop_holds(current_state, [kettle, burner1])
    kettle_on_burner2 = env._OnTop_holds(current_state, [kettle, burner2])
    kettle_on_burner3 = env._OnTop_holds(current_state, [kettle, burner3])
    kettle_on_burner4 = env._OnTop_holds(current_state, [kettle, burner4])
    
    print(f"壶在炉子1上: {kettle_on_burner1}")
    print(f"壶在炉子2上: {kettle_on_burner2}")
    print(f"壶在炉子3上: {kettle_on_burner3}")
    print(f"壶在炉子4上: {kettle_on_burner4}")
    
    # 检查旋钮状态
    knob1_on = env.On_holds(current_state, [knob1])
    knob2_on = env.On_holds(current_state, [knob2])
    knob3_on = env.On_holds(current_state, [knob3])
    knob4_on = env.On_holds(current_state, [knob4])
    light_on = env.On_holds(current_state, [light])
    
    print(f"\n旋钮1开启: {knob1_on}")
    print(f"旋钮2开启: {knob2_on}")
    print(f"旋钮3开启: {knob3_on}")
    print(f"旋钮4开启: {knob4_on}")
    print(f"灯开启: {light_on}")
    
    # 检查壶是否沸腾
    kettle_boiling1 = env._KettleBoiling_holds(current_state, [kettle, burner1, knob1])
    kettle_boiling2 = env._KettleBoiling_holds(current_state, [kettle, burner2, knob2])
    kettle_boiling3 = env._KettleBoiling_holds(current_state, [kettle, burner3, knob3])
    kettle_boiling4 = env._KettleBoiling_holds(current_state, [kettle, burner4, knob4])
    
    print(f"\n壶在炉子1上沸腾: {kettle_boiling1}")
    print(f"壶在炉子2上沸腾: {kettle_boiling2}")
    print(f"壶在炉子3上沸腾: {kettle_boiling3}")
    print(f"壶在炉子4上沸腾: {kettle_boiling4}")
    
    # 显示物件角度信息
    print(f"\n=== 物件角度信息 ===")
    for obj in [knob1, knob2, knob3, knob4, light]:
        if obj in current_state:
            angle = current_state[obj].get("angle", "N/A")
            print(f"{obj.name} 角度: {angle:.4f}")
    
    print(f"\n=== 环境总结 ===")
    print(f"当前任务: {env._current_task.goal_description}")
    print(f"目标是否达成: {env.goal_reached()}")
    
else:
    print("环境未初始化，请先运行第一个cell")
